# 0. Setup

## Librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq

# 1. Datos

## 1.1 Cargar Dataset

Cargar dataset. Se utilizan chunks para ahorrar RAM y luego se guarda en formato parquet.

In [11]:
ruta_csv = 'global_concatenado/global_concatenado.CSV'
source_parquet = 'global_concatenado/global_concatenado.parquet'

In [ ]:
reader = pd.read_csv(
    ruta_csv,
    chunksize=100_000,
    low_memory=False
)

writer = None

for i, chunk in enumerate(reader):
    print(f"Processing chunk {i}")

    float_cols = chunk.select_dtypes(include=['float64']).columns
    int_cols = chunk.select_dtypes(include=['int64']).columns

    chunk[float_cols] = chunk[float_cols].astype('float32')
    chunk[int_cols] = chunk[int_cols].apply(pd.to_numeric, downcast='unsigned')

    table = pa.Table.from_pandas(chunk)

    if writer is None:
        writer = pq.ParquetWriter(
            source_parquet,
            table.schema
        )

    writer.write_table(table)

if writer:
    writer.close()

print("Saved parquet successfully.")

Processing chunk 0
Processing chunk 1
Processing chunk 2
Processing chunk 3
Processing chunk 4
Processing chunk 5
Processing chunk 6
Processing chunk 7
Processing chunk 8
Processing chunk 9
Processing chunk 10
Processing chunk 11
Processing chunk 12
Processing chunk 13
Processing chunk 14
Processing chunk 15
Processing chunk 16
Processing chunk 17
Processing chunk 18
Processing chunk 19
Processing chunk 20
Processing chunk 21
Processing chunk 22
Processing chunk 23
Processing chunk 24
Processing chunk 25
Processing chunk 26
Processing chunk 27
Processing chunk 28
Processing chunk 29
Processing chunk 30
Processing chunk 31
Processing chunk 32
Processing chunk 33
Processing chunk 34
Processing chunk 35
Processing chunk 36
Processing chunk 37
Processing chunk 38
Processing chunk 39
Processing chunk 40
Processing chunk 41
Processing chunk 42
Processing chunk 43
Processing chunk 44
Processing chunk 45
Processing chunk 46
Processing chunk 47
Processing chunk 48
Processing chunk 49
Processing

Luego cargamos desde parquet

In [12]:
df = pd.read_parquet(source_parquet)

print(df.info(memory_usage='deep'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8644192 entries, 0 to 8644191
Columns: 109 entries, Unnamed: 0.1 to rst_psa6
dtypes: float32(104), object(3), uint16(1), uint32(1)
memory usage: 5.0 GB
None


## 1.2 Preprocesado

### Dimensiones y presencia de nulos

In [13]:
n_rows, n_cols = df.shape

print(f"Num. de observaciones (rows): {n_rows}")
print(f"Num. variables (columns): {n_cols}")
print(f"Existen nulos: {df.isnull().values.any()}")

Num. de observaciones (rows): 8644192
Num. variables (columns): 109
Existen nulos: True


In [14]:
# Crear un DataFrame de resumen con nulos y tipos de dato
resumen = pd.DataFrame({
    'Nulos': df.isnull().sum(),
    'Tipo_de_Dato': df.dtypes
})

# Forzar a que se muestren todas las filas
print(resumen.to_string())

                       Nulos Tipo_de_Dato
Unnamed: 0.1               0       uint32
Unnamed: 0                 0       uint16
source                     0       object
psi_psa1              297228      float32
psi_psa2              297228      float32
psi_psa3              434287      float32
psi_psa4              434287      float32
psi_tablero           297228      float32
flujo                 280060      float32
totalizador           297228      float32
TIME                       0       object
r_psa1                  7649      float32
r_psa2                  7649      float32
r_psa3                144708      float32
r_psa4                144708      float32
r_gen1                 21204      float32
r_gen2                 21204      float32
r_bar                  21204      float32
r_sec1                 21204      float32
r_sec2                 21204      float32
r_com1                 21204      float32
r_com2                 21204      float32
r_com3                158263      


### **¿Qué significa cada columna? **

Sabiendo que la empresa se dedica a inyectar oxígeno a los peces las empresas utilizan plantas generadoras de oxígeno in situ.

Como tienes 109 columnas, la mejor forma de entenderlas es agruparlas por su **prefijo**, ya que siguen una nomenclatura estándar de telemetría industrial (sistemas SCADA o PLCs).

**1. Identificadores y Tiempo**

* `Unnamed: 0` / `Unnamed: 0.1`: Son índices antiguos o números de fila que se guardaron por error al exportar el CSV originalmente. (Te sugiero eliminarlas después).
* `source`: El origen de los datos o el identificador del pontón/centro de cultivo (ej. POXC1).
* `TIME`: La marca de tiempo (fecha y hora) exacta en la que se tomó la medición.
* `Sistema`: Probablemente el nombre o ID general del sistema operando.

**2. Sistema PSA (Generadores de Oxígeno)**
*Las plantas generan oxígeno separándolo del aire mediante un proceso llamado PSA (Pressure Swing Adsorption).*

* `psi_psa1` a `psi_psa6`: La presión (en PSI - libras por pulgada cuadrada) de cada uno de los generadores PSA (hasta 6 equipos).
* `suma_psa`: Cantidad total de módulos PSA que están encendidos o funcionando en ese momento.

**3. Flujo y Entrega de Oxígeno**

* `psi_tablero`: La presión de oxígeno en el tablero principal de distribución, justo antes de enviarlo a las jaulas de los peces.
* `flujo`: La cantidad de oxígeno que se está inyectando en ese instante (probablemente medido en litros por minuto o metros cúbicos por hora).
* `totalizador`: El volumen acumulado total de oxígeno que se ha entregado a lo largo del tiempo (como el cuentakilómetros de un auto).

**4. Variables de Estado o Funcionamiento (`r_`)**
*La "r" generalmente viene de "Run" (en marcha/corriendo) o "Relay". Indican si un equipo está encendido (1) o apagado (0).*

* `r_psa1` a `r_psa6`: Estado de marcha de los módulos PSA.
* `r_com1` a `r_com4`: Estado de marcha de los compresores de aire (el aire comprimido alimenta a los PSA).
* `suma_compresores`: Cantidad total de compresores encendidos.
* `r_gen1`, `r_gen2`: Estado de los generadores eléctricos.
* `r_sec1`, `r_sec2`: Estado de los secadores de aire (eliminan la humedad del aire antes de que entre al PSA).
* `r_bar`: Estado de la barredora o sistema de barrido.

**5. Variables de Control (`sp_`, `ox_`, `m_`)**
*Datos provenientes de los sensores instalados (probablemente en las distintas jaulas o líneas de inyección de oxígeno, numerados del s1 al s12).*

* `sp_s1` a `sp_s12`: **Setpoint** (Punto de ajuste). Es el nivel de oxígeno o presión objetivo que el operador programó en el sistema para ese sensor.
* `ox_s1` a `ox_s12`: La medición real de **concentración o nivel de oxígeno** (pureza) que está leyendo el sensor.
* `m_s1` a `m_s12`: Probablemente **Modo** de operación (ej. Automático vs Manual) o estado de la válvula/caudalímetro de ese sensor.

**6. Temperatura**

* `mb_g1_temperatura_f` / `mb_g2_temperatura_f`: La temperatura en grados Fahrenheit (°F) de los generadores o motores 1 y 2 (el prefijo "mb" suele referirse a Modbus, el protocolo de comunicación utilizado para extraer el dato).

**7. Señales del Sistema (`hb_`, `rst_`)**

* `hb_*` (ej. `hb_psa1`, `hb_com1`): **Heartbeat** (Latido). Es una señal digital (1 o 0) que el equipo envía cada segundo para decir "Estoy conectado y en línea". Si se pierde, significa que se cortó la comunicación de internet o red con el equipo.
* `rst_*` (ej. `rst_psa1`, `rst_com1`): **Reset / Restart**. Indica si se envió una señal de reinicio a ese equipo, o la cantidad de veces que se ha reiniciado por fallas.

In [15]:
# Condición: Que el valor NO (~) esté en la lista [0, 1] para alguna de las 3 columnas
condicion = (
    (~df['m_s1'].isin([0.0, 1.0])) | 
    (~df['m_s2'].isin([0.0, 1.0])) | 
    (~df['m_s3'].isin([0.0, 1.0]))
)

# Filtramos las columnas, quitamos los nulos
df_filtrado = df.loc[condicion, ['m_s1', 'm_s2', 'm_s3', 'ox_s1']].dropna()

# Vemos cuántos datos "raros" hay y mostramos los primeros 10
print(f"Se encontraron {len(df_filtrado)} filas con valores distintos de 0 y 1.\n")

if len(df_filtrado) > 0:
    print(df_filtrado.head(10))
else:
    print("¡Confirmado! Todos los datos en esas columnas son estrictamente 0.0 o 1.0")

Se encontraron 0 filas con valores distintos de 0 y 1.

¡Confirmado! Todos los datos en esas columnas son estrictamente 0.0 o 1.0


In [16]:
ruta_salida = "global_concatenado/df_filtrado.parquet"

df_filtrado.to_parquet(
    ruta_salida,
    engine="pyarrow",
    index=False
)

print(f"DataFrame guardado en: {ruta_salida}")

DataFrame guardado en: global_concatenado/df_filtrado.parquet
